# VK Ads: прогноз без data leakage

Итоговый эксперимент разделён на три временные зоны:

1. **development** — расчёт устойчивой калибровки;
2. **calibration** — выбор числа лагов, весов blend и необходимости bias;
3. **final holdout** — однократная оценка уже зафиксированного прогноза.

Кампания допускается в train только при `hour_end < следующий cutoff`.
История для прогнозирования validation обрезается перед первым `hour_start`.

In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = Path("/Users/anastasiasergeeva/Documents/New project/vk_ads_coursework")
DATA_DIR = Path(os.environ.get(
    "VK_ADS_DATA_DIR",
    "/Users/anastasiasergeeva/Desktop/HSE/Сессия 2026/НИР Vk Ads/data",
))
OUTPUT_DIR = PROJECT_DIR / "outputs"
sys.path.insert(0, str(PROJECT_DIR / "src"))

from vk_ads_solution import (
    AuctionReplayForecaster,
    build_leak_free_campaign_features,
    load_dataset,
    purged_three_way_split,
    smoothed_mean_log_accuracy_ratio,
)

## 1. Данные и as-of cutoff

In [2]:
users, history, validate, answers = load_dataset(DATA_DIR)
forecast_cutoff = int(validate["hour_start"].min())
past_history = history.loc[history["hour"] < forecast_cutoff].copy()
split = purged_three_way_split(
    validate, development_fraction=0.6, calibration_fraction=0.2
)

audit = pd.DataFrame([{
    "history_end": int(past_history["hour"].max()),
    "forecast_cutoff": forecast_cutoff,
    "calibration_start": split.calibration_start,
    "test_start": split.test_start,
    "development_rows": len(split.development_idx),
    "calibration_rows": len(split.calibration_idx),
    "pretest_rows": len(split.pretest_idx),
    "final_holdout_rows": len(split.test_idx),
    "max_development_end": int(validate.iloc[split.development_idx]["hour_end"].max()),
    "max_pretest_end": int(validate.iloc[split.pretest_idx]["hour_end"].max()),
}])
display(audit)
assert audit.at[0, "history_end"] < audit.at[0, "forecast_cutoff"]
assert audit.at[0, "max_development_end"] < audit.at[0, "calibration_start"]
assert audit.at[0, "max_pretest_end"] < audit.at[0, "test_start"]

,history_end,forecast_cutoff,calibration_start,test_start,development_rows,calibration_rows,pretest_rows,final_holdout_rows,max_development_end,max_pretest_end
0,746,747,1107,1257,389,121,643,201,1106,1256


## 2. Архитектура

Для каждого объявления replay выполняется только на полностью доступных прошлых окнах:

- одно 31-дневное monthly-окно;
- несколько daily-окон с тем же часом суток;
- несколько weekly-окон с тем же часом недели.

В каждом окне применяются правила CPM-аукциона, шестичасовые сессии и
Poisson-binomial агрегация `P(1+)`, `P(2+)`, `P(3+)`. Число лагов и веса
геометрического blend выбраны только по кампаниям, полностью завершённым
до final holdout (`pretest`).

## 3. Проверка зафиксированной модели

In [3]:
lock_path = OUTPUT_DIR / "strict_model_lock.json"
prediction_path = OUTPUT_DIR / "strict_locked_predictions.tsv"
manifest = json.loads(lock_path.read_text(encoding="utf-8"))
prediction_hash = hashlib.sha256(prediction_path.read_bytes()).hexdigest()
assert prediction_hash == manifest["prediction_sha256"]
assert manifest["final_holdout_metric"] is None
assert manifest["test_start"] == split.test_start

locked_prediction = pd.read_csv(prediction_path, sep="	")
selection = pd.read_csv(OUTPUT_DIR / "strict_calibration_selection.csv")
display(pd.DataFrame([manifest]))
display(selection.head(10))

,protocol,model_family,forecast_cutoff,history_end,calibration_start,test_start,development_rows,calibration_rows,pretest_rows,final_holdout_rows,...,weekly_weight,target_configs,use_bias,pretest_bias,selection_metric,calibration_metric_percent,pretest_metric_percent,prediction_file,prediction_sha256,final_holdout_metric
0,purged_development_calibration_locked_final_ho...,scalar_past_only_blend,747,746,1107,1257,389,121,643,201,...,0.55,None,False,"{'at_least_one': 0.0, 'at_least_two': 0.0, 'at...",pretest_metric_percent,8.03,8.17,strict_locked_predictions.tsv,e17b96a74a8e795dda76610abb8239c666ad1f7e18ac1b...,None


,candidate_id,model_family,target_configs,daily_lags,weekly_lags,monthly_weight,daily_weight,weekly_weight,use_bias,calibration_metric_percent,pretest_metric_percent
0,scalar_d8_w5_weights_0.05_0.40_0.55,scalar_past_only_blend,NaN,8.0,5.0,0.05,0.40,0.55,False,8.03,8.17
1,scalar_d8_w6_weights_0.05_0.40_0.55,scalar_past_only_blend,NaN,8.0,6.0,0.05,0.40,0.55,False,8.03,8.17
2,scalar_d8_w5_weights_0.05_0.35_0.60,scalar_past_only_blend,NaN,8.0,5.0,0.05,0.35,0.60,False,8.02,8.18
3,scalar_d8_w6_weights_0.05_0.35_0.60,scalar_past_only_blend,NaN,8.0,6.0,0.05,0.35,0.60,False,8.02,8.18
4,scalar_d8_w5_weights_0.05_0.35_0.60,scalar_past_only_blend,NaN,8.0,5.0,0.05,0.35,0.60,True,8.03,8.18
5,scalar_d8_w5_weights_0.05_0.40_0.55,scalar_past_only_blend,NaN,8.0,5.0,0.05,0.40,0.55,True,8.03,8.18
6,scalar_d8_w6_weights_0.05_0.35_0.60,scalar_past_only_blend,NaN,8.0,6.0,0.05,0.35,0.60,True,8.03,8.18
7,scalar_d8_w6_weights_0.05_0.40_0.55,scalar_past_only_blend,NaN,8.0,6.0,0.05,0.40,0.55,True,8.03,8.18
8,scalar_d7_w5_weights_0.10_0.35_0.55,scalar_past_only_blend,NaN,7.0,5.0,0.10,0.35,0.55,False,8.04,8.18
9,scalar_d7_w6_weights_0.10_0.35_0.55,scalar_past_only_blend,NaN,7.0,6.0,0.10,0.35,0.55,False,8.04,8.18


В lock-файле нет final-метрики: сначала `select_strict_model.py` записал конфигурацию
и SHA-256 прогноза. Только после этого отдельный `evaluate_locked_holdout.py` получил
доступ к ответам final-блока.

In [4]:
final_answers = answers.iloc[split.test_idx].reset_index(drop=True)
final_prediction = locked_prediction.iloc[split.test_idx].reset_index(drop=True)
final_metric = smoothed_mean_log_accuracy_ratio(final_answers, final_prediction)

strict_result = pd.DataFrame([{
    "protocol": manifest["protocol"],
    "selected_candidate": manifest["selected_candidate"],
    "calibration_metric_percent": manifest["calibration_metric_percent"],
    "pretest_metric_percent": manifest["pretest_metric_percent"],
    "final_holdout_rows": len(split.test_idx),
    "final_holdout_metric_percent": final_metric,
}])
display(strict_result)

,protocol,selected_candidate,calibration_metric_percent,pretest_metric_percent,final_holdout_rows,final_holdout_metric_percent
0,purged_development_calibration_locked_final_ho...,scalar_d8_w5_weights_0.05_0.40_0.55,8.03,8.17,201,9.54


## 4. Декомпозиционные агрегаты и индикаторы

In [5]:
past_model = AuctionReplayForecaster(session_gap_hours=6).fit(past_history)
features = build_leak_free_campaign_features(
    validate, users, past_model, source_shift=24 * 31
)
print("Feature matrix:", features.shape)
display(features.filter(regex="missing|zero").sum().sort_values().tail(12))

Feature matrix: (1008, 114)


tie_independent_at_least_one_zero       32.0
strict_at_least_one_zero                68.0
supply_ceiling_at_least_two_zero       273.0
tie_half_at_least_two_zero             328.0
tie_full_at_least_two_zero             328.0
tie_independent_at_least_two_zero      328.0
strict_at_least_two_zero               350.0
supply_ceiling_at_least_three_zero     423.0
tie_independent_at_least_three_zero    472.0
tie_full_at_least_three_zero           472.0
tie_half_at_least_three_zero           472.0
strict_at_least_three_zero             488.0
dtype: float64

## Результат

- Calibration diagnostic: **8.03%**.
- Selection по pretest: **8.17%**.
- Locked final temporal holdout: **9.54%** на 201 строке.
- Финальная конфигурация: `daily=8`, `weekly=5`, веса `0.05 / 0.40 / 0.55`, без bias.
- Prediction API не читает `validate_answers.tsv` и отклоняет кампании,
  пересекающиеся с доступной историей.

Это основная честная offline-метрика. Значения на всём открытом validation можно
использовать только как development diagnostics, но не как независимую оценку качества.